# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ruchi74/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method Choice and Justification

I selected a Decision Tree Classifier because it is simple to implement, easy to interpret, and suitable for binary classification. It can identify patterns between content features and the target variable without requiring complex preprocessing or model architecture.

The target is defined as 1 when impressions in the last 30 days are at least 20% lower than impressions in the previous 30 days, and 0 otherwise. The model uses content age, session activity, engagement, and other available features to classify the target.

I limited the tree depth to 5 and set a minimum of 50 samples per leaf to reduce overfitting. I used a stratified 80/20 train-test split to maintain similar target proportions in both sets.

In [4]:

import pandas as pd
import numpy as np
from pathlib import Path

# Load the FlyRank starter dataset
data_url = (
    "https://raw.githubusercontent.com/"
    "Ruchi74/flyrank-ml-internship/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_url)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


Dataset loaded successfully!
Dataset shape: (30000, 44)

Column names:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [ ]:

# STEP 3 — CHECK AVAILABLE TARGET LABELS

print("Dataset columns that may contain labels or outcomes:")
for col in df.columns:
    if any(word in col.lower() for word in ["label", "target", "declin", "trend"]):
        print("-", col)

print("\nTrend direction counts (inspection only):")
print(df["trend_direction"].value_counts(dropna=False))

print("\nTrend percentage summary (inspection only):")
print(df["trend_pct"].describe())

print("\nRecent and previous period columns:")
print([
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d"
])


Dataset columns that may contain labels or outcomes:
- trend_direction
- trend_pct

Trend direction counts (inspection only):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Trend percentage summary (inspection only):
count    26612.000000
mean        -4.785969
std        473.861780
min       -100.000000
25%        -62.600000
50%        -33.500000
75%          0.000000
max      44900.000000
Name: trend_pct, dtype: float64

Recent and previous period columns:
['impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d']


In [5]:
# STEP 4 — DEFINE THE TARGET

model_df = df[df["trend_direction"].notna()].copy()

model_df["is_declining_label"] = (
    model_df["trend_direction"] == "down"
).astype(int)

print("Rows available for modeling:", len(model_df))
print("\nTarget counts:")
print(model_df["is_declining_label"].value_counts().sort_index())

print("\nTarget proportions:")
print(
    model_df["is_declining_label"]
    .value_counts(normalize=True)
    .sort_index()
    .round(3)
)

print("\nTarget definition:")
print("1 = trend_direction is down")
print("0 = trend_direction is not down")


Rows available for modeling: 30000

Target counts:
is_declining_label
0    13738
1    16262
Name: count, dtype: int64

Target proportions:
is_declining_label
0    0.458
1    0.542
Name: proportion, dtype: float64

Target definition:
1 = trend_direction is down
0 = trend_direction is not down


In [15]:


# STEP 5 — CLIENT-GROUPED TRAIN/TEST SPLIT

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer

excluded_columns = [
    "content_id", "client_id", "is_declining_label",
    "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
    "ctr", "impressions_90d", "clicks_90d",
    "days_with_impressions", "impression_tier"
]

X = model_df.drop(columns=excluded_columns, errors="ignore")
y = model_df["is_declining_label"]
groups = model_df["client_id"]

X = pd.get_dummies(X, drop_first=True, dtype=int)
X = X.replace([np.inf, -np.inf], np.nan)

# Split by client so clients cannot appear in both sets
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

# Fit imputer only on training data
imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X.columns,
    index=X_train.index
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X.columns,
    index=X_test.index
)

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Clients in both sets:", len(train_clients & test_clients))
print("Features:", X_train.shape[1])
print("Missing values in training:", X_train.isna().sum().sum())
print("Missing values in testing:", X_test.isna().sum().sum())

Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Clients in both sets: 0
Features: 47
Missing values in training: 0
Missing values in testing: 0


In [ ]:


# STEP 6 — CHECK FOR TARGET LEAKAGE

print("Target definition:")
print("Decline = last 30-day impressions <= 80% of previous 30-day impressions")

print("\nExcluded target-related columns:")
print([
    "impressions_last_30d",
    "impressions_prev_30d",
    "trend_direction",
    "trend_pct"
])

print("\nAre target-related columns in model features?")
for col in [
    "impressions_last_30d",
    "impressions_prev_30d",
    "trend_direction",
    "trend_pct"
]:
    print(f"{col}: {col in X_train.columns}")

print("\nLeakage check complete.")



Target definition:
Decline = last 30-day impressions <= 80% of previous 30-day impressions

Excluded target-related columns:
['impressions_last_30d', 'impressions_prev_30d', 'trend_direction', 'trend_pct']

Are target-related columns in model features?
impressions_last_30d: False
impressions_prev_30d: False
trend_direction: False
trend_pct: False

Leakage check complete.


In [7]:


# STEP 7 — RETRAIN THE DECISION TREE

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("UPDATED DECISION TREE RESULTS")
print("-----------------------------")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 3))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 3))
print("F1 Score:", round(f1_score(y_test, y_pred, zero_division=0), 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


UPDATED DECISION TREE RESULTS
-----------------------------
Accuracy: 0.641
Precision: 0.655
Recall: 0.713
F1 Score: 0.683

Confusion Matrix:
[[1524 1224]
 [ 932 2320]]


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Train-Test Split Design

The dataset was divided into 80% training data and 20% testing data using `train_test_split` with `random_state=42` and stratification.

- **Total rows used:** 18,010
- **Training rows:** 14,408
- **Testing rows:** 3,602
- **Training features:** 47
- **Training target proportions:** 61.6% decline and 38.4% non-decline
- **Testing target proportions:** 61.6% decline and 38.4% non-decline

Missing feature values were imputed using median values calculated from the training set only. The same imputer was then applied to the test set. This avoids using test-set information to calculate imputation values.

The target is based on historical impression changes, so this split evaluates classification of that retrospective target, not prediction of future declines.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Train and Compare Against the Week 4 Baseline

I trained a Decision Tree Classifier using the training dataset and evaluated it on the held-out test dataset. The model used a maximum depth of 5, a minimum of 50 samples per leaf, and balanced class weights.

The model was evaluated using accuracy, precision, recall, and F1 score.

| Method | Accuracy | Precision | Recall | F1 Score |
|---|---:|---:|---:|---:|
| Week 4 Baseline | 0.526 | 0.639 | 0.533 | 0.581 |
| Decision Tree | 0.669 | 0.743 | 0.707 | 0.725 |

The Decision Tree achieved an F1 score of 0.725, compared with 0.581 for the Week 4 baseline. Its accuracy, precision, recall, and F1 score were all higher in this evaluation.

**Limitation:** The Week 4 baseline identifies pages with low click-through rates, while the Decision Tree classifies historical impression declines. Therefore, this is a preliminary comparison rather than a fully like-for-like evaluation. The results do not establish that the model can predict future declines.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Error Analysis — Client-Grouped Test Set

The Decision Tree made 2,842 incorrect predictions out of 6,163 test records.

1. **False Negative:** The actual trend was down, with a trend percentage of −57.7%, but the model predicted not declining. The content was 445 days old.
2. **False Positive:** The actual trend was stable, with a trend percentage of 10.4%, but the model predicted declining. The content was 238 days old.
3. **False Negative:** The actual trend was down, with a trend percentage of −100%, but the model predicted not declining. The content was 502 days old.

**Conclusion:** The model misses some strongly declining content and incorrectly flags some stable content. Its performance on unseen clients is limited, so predictions should be reviewed before making business decisions.

### Error Analysis

The Decision Tree made 2,156 incorrect predictions out of 6,000 test records.

1. **Example 1 — False Positive:** The actual trend was stable, but the model predicted declining. The content had an average position of 15.0 and was 223 days old.
2. **Example 2 — False Negative:** The actual trend was down, but the model predicted not declining. The content had an average position of 15.8 and was 463 days old.
3. **Example 3 — False Positive:** The actual trend was new, but the model predicted declining. The trend percentage was unavailable, and the content had an average position of 18.3.

**Conclusion:** The model sometimes confuses stable or new content with declining content and can miss genuinely declining content. These examples show why individual prediction errors should be reviewed before using the model for business decisions.

### Feature Importance Interpretation

The Decision Tree model identified `content_age_days` as the most important feature, followed by `days_with_sessions`, `scroll_rate`, and `avg_position`. These features contributed most to the model's decisions. Some features had zero importance because the tree did not use them to make splits.

Feature importance indicates predictive contribution, not causation. The model achieved an F1 score of 0.725 and accuracy of 66.9% on the test set. Since the target was created from historical impression changes, these results should be treated as a retrospective proxy exercise rather than evidence of reliable future prediction.

### Model vs. Week 4 Baseline

The Decision Tree achieved an accuracy of 0.669 and an F1 score of 0.725. The Week 4 baseline achieved an accuracy of 0.526 and an F1 score of 0.581.

The Decision Tree performed better on the evaluated test set across accuracy, precision, recall, and F1 score. Its F1 score was 0.144 higher than the baseline.

However, the Week 4 baseline was originally designed to identify low-CTR pages rather than predict impression declines. Therefore, this comparison is preliminary and should not be interpreted as a fully equivalent comparison. The target is based on historical impression changes, so future predictive performance has not been established.

### Limitations and Error Analysis

The Decision Tree achieved 66.9% accuracy and an F1 score of 0.725 on the test set. The confusion matrix shows 1,570 true positives, 840 true negatives, 542 false positives, and 650 false negatives.

False positives are pages classified as declining even though they did not meet the defined decline threshold. False negatives are pages that met the decline threshold but were classified as non-declining.

The model's target is calculated from historical impression windows. Therefore, the results do not establish the ability to predict future declines. Also, the Week 4 baseline was designed to flag low-CTR pages, not impression declines. Its comparison with this model is preliminary and is not a fully like-for-like evaluation.

Future work should use a clearly defined future outcome window and compare the model with a baseline designed to predict the same outcome.

In [ ]:




# STEP 8 — UPDATED BASELINE COMPARISON

baseline_pred = (
    (model_df.loc[X_test.index, "impressions_90d"] >= 500) &
    (model_df.loc[X_test.index, "avg_position"] > 0) &
    (model_df.loc[X_test.index, "avg_position"] <= 20) &
    (model_df.loc[X_test.index, "ctr"] < 0.5)
).astype(int)

comparison = pd.DataFrame({
    "Method": ["Week 4 Baseline", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, baseline_pred),
        accuracy_score(y_test, y_pred)
    ],
    "Precision": [
        precision_score(y_test, baseline_pred, zero_division=0),
        precision_score(y_test, y_pred, zero_division=0)
    ],
    "Recall": [
        recall_score(y_test, baseline_pred, zero_division=0),
        recall_score(y_test, y_pred, zero_division=0)
    ],
    "F1 Score": [
        f1_score(y_test, baseline_pred, zero_division=0),
        f1_score(y_test, y_pred, zero_division=0)
    ]
})

display(comparison.round(3))


,Method,Accuracy,Precision,Recall,F1 Score
0,Week 4 Baseline,0.526,0.639,0.533,0.581
1,Decision Tree,0.669,0.743,0.707,0.725


In [8]:
# STEP 8 — FEATURE IMPORTANCE

import pandas as pd

importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("TOP 10 IMPORTANT FEATURES")
display(importance_df.head(10))

TOP 10 IMPORTANT FEATURES


,Feature,Importance
15,avg_position,0.433665
12,content_age_days,0.254644
11,days_with_sessions,0.085523
14,days_since_last_update,0.049248
5,pageviews_90d,0.046902
17,scroll_rate,0.043868
10,scroll_events_90d,0.025503
29,model_used_gpt-5-mini,0.024693
28,model_used_gpt-4o-mini,0.016158
27,model_used_gemini-3-flash-preview,0.013336


In [9]:
# STEP 18 — BASELINE MODEL COMPARISON

from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score
)

# Simple baseline: always predict the majority class
majority_class = y_train.mode()[0]

baseline_pred = [majority_class] * len(y_test)

print("BASELINE RESULTS")
print("----------------")
print("Baseline prediction:", majority_class)
print("Accuracy:", round(accuracy_score(y_test, baseline_pred), 3))
print("Precision:", round(precision_score(
    y_test, baseline_pred, zero_division=0
), 3))
print("Recall:", round(recall_score(
    y_test, baseline_pred, zero_division=0
), 3))
print("F1 Score:", round(f1_score(
    y_test, baseline_pred, zero_division=0
), 3))

print("\nDECISION TREE RESULTS")
print("---------------------")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(
    y_test, y_pred, zero_division=0
), 3))
print("Recall:", round(recall_score(
    y_test, y_pred, zero_division=0
), 3))
print("F1 Score:", round(f1_score(
    y_test, y_pred, zero_division=0
), 3))

BASELINE RESULTS
----------------
Baseline prediction: 1
Accuracy: 0.542
Precision: 0.542
Recall: 1.0
F1 Score: 0.703

DECISION TREE RESULTS
---------------------
Accuracy: 0.641
Precision: 0.655
Recall: 0.713
F1 Score: 0.683


In [10]:
# STEP 19 — ERROR ANALYSIS

errors = X_test.copy()
errors["Actual"] = y_test
errors["Predicted"] = y_pred

wrong_predictions = errors[
    errors["Actual"] != errors["Predicted"]
]

print("Total incorrect predictions:", len(wrong_predictions))
print("\nThree incorrect examples:")
display(wrong_predictions.head(3))

Total incorrect predictions: 2156

Three incorrect examples:


,search_volume,competition,cpc,word_count,char_count,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,...,word_count_tier_<1000,char_count_tier_25000+,char_count_tier_8000-15000,char_count_tier_<8000,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3,Actual,Predicted
2110,0.0,0.00,0.0,6466.0,42912.0,66.0,41.0,41.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0,1
27890,10.0,0.91,0.0,2877.0,19135.0,13.0,12.0,12.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1,0
9902,20.0,0.00,0.0,2892.0,19032.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0,1


In [11]:
# STEP 20 — READABLE ERROR ANALYSIS

error_details = model_df.loc[
    wrong_predictions.index,
    [
        "trend_direction",
        "trend_pct",
        "content_type",
        "avg_position",
        "content_age_days",
        "days_with_sessions",
        "days_since_last_update"
    ]
].copy()

error_details["Actual"] = wrong_predictions["Actual"]
error_details["Predicted"] = wrong_predictions["Predicted"]

display(error_details.head(3))

,trend_direction,trend_pct,content_type,avg_position,content_age_days,days_with_sessions,days_since_last_update,Actual,Predicted
2110,stable,19.1,keyword article,15.0,223,15,104,0,1
27890,down,-83.2,keyword article,15.8,463,11,22,1,0
9902,new,NaN,keyword article,18.3,175,1,20,0,1


In [12]:
# STEP 22 — LEAKAGE CHECK

print("trend_direction in features:", "trend_direction" in X_train.columns)
print("trend_pct in features:", "trend_pct" in X_train.columns)
print("content_id in features:", "content_id" in X_train.columns)
print("client_id in features:", "client_id" in X_train.columns)# STEP 22 — LEAKAGE CHECK

print("trend_direction in features:", "trend_direction" in X_train.columns)
print("trend_pct in features:", "trend_pct" in X_train.columns)
print("content_id in features:", "content_id" in X_train.columns)
print("client_id in features:", "client_id" in X_train.columns)

trend_direction in features: False
trend_pct in features: False
content_id in features: False
client_id in features: False
trend_direction in features: False
trend_pct in features: False
content_id in features: False
client_id in features: False


In [13]:
# STEP 23 — REPRODUCIBILITY CHECK

print("Train-test split random_state: 42")
print("Decision Tree random_state:", model.random_state)
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Number of features:", X_train.shape[1])
print("Target column: is_declining_label")

Train-test split random_state: 42
Decision Tree random_state: 42
Training rows: 24000
Testing rows: 6000
Number of features: 47
Target column: is_declining_label


In [14]:
# STEP 24 — CHECK CLIENT GROUPS

print("Unique clients:", model_df["client_id"].nunique())
print("Total records:", len(model_df))
print("\nRecords per client:")
print(model_df.groupby("client_id").size().describe())

Unique clients: 32
Total records: 30000

Records per client:
count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
dtype: float64


In [16]:
# STEP 26 — RETRAIN MODEL ON GROUPED SPLIT

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)

model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("GROUPED-SPLIT DECISION TREE RESULTS")
print("-----------------------------------")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(
    y_test, y_pred, zero_division=0
), 3))
print("Recall:", round(recall_score(
    y_test, y_pred, zero_division=0
), 3))
print("F1 Score:", round(f1_score(
    y_test, y_pred, zero_division=0
), 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

GROUPED-SPLIT DECISION TREE RESULTS
-----------------------------------
Accuracy: 0.539
Precision: 0.541
Recall: 0.643
F1 Score: 0.588

Confusion Matrix:
[[1295 1719]
 [1123 2026]]


In [17]:
# STEP 27 — BASELINE ON GROUPED SPLIT

from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score
)

majority_class = y_train.mode()[0]
baseline_pred = np.full(len(y_test), majority_class)

print("GROUPED-SPLIT BASELINE RESULTS")
print("--------------------------------")
print("Baseline prediction:", majority_class)
print("Accuracy:", round(accuracy_score(y_test, baseline_pred), 3))
print("Precision:", round(precision_score(
    y_test, baseline_pred, zero_division=0
), 3))
print("Recall:", round(recall_score(
    y_test, baseline_pred, zero_division=0
), 3))
print("F1 Score:", round(f1_score(
    y_test, baseline_pred, zero_division=0
), 3))

print("\nGROUPED-SPLIT DECISION TREE RESULTS")
print("-----------------------------------")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(
    y_test, y_pred, zero_division=0
), 3))
print("Recall:", round(recall_score(
    y_test, y_pred, zero_division=0
), 3))
print("F1 Score:", round(f1_score(
    y_test, y_pred, zero_division=0
), 3))

GROUPED-SPLIT BASELINE RESULTS
--------------------------------
Baseline prediction: 1
Accuracy: 0.511
Precision: 0.511
Recall: 1.0
F1 Score: 0.676

GROUPED-SPLIT DECISION TREE RESULTS
-----------------------------------
Accuracy: 0.539
Precision: 0.541
Recall: 0.643
F1 Score: 0.588


In [18]:
# STEP 28 — UPDATED FEATURE IMPORTANCE

importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

print("TOP 10 FEATURES — GROUPED-SPLIT MODEL")
display(importance_df.head(10))

TOP 10 FEATURES — GROUPED-SPLIT MODEL


,Feature,Importance
15,avg_position,0.407196
12,content_age_days,0.239876
14,days_since_last_update,0.087709
11,days_with_sessions,0.074281
0,search_volume,0.051937
41,char_count_tier_8000-15000,0.045303
17,scroll_rate,0.044405
28,model_used_gpt-4o-mini,0.024058
29,model_used_gpt-5-mini,0.022332
16,engagement_rate,0.002636


In [19]:
# STEP 29 — ERROR ANALYSIS FOR GROUPED SPLIT

errors = X_test.copy()
errors["Actual"] = y_test
errors["Predicted"] = y_pred

wrong_predictions = errors[
    errors["Actual"] != errors["Predicted"]
]

print("Incorrect predictions:", len(wrong_predictions))
print("\nThree incorrect examples:")

error_details = model_df.loc[
    wrong_predictions.index,
    [
        "trend_direction",
        "trend_pct",
        "content_type",
        "avg_position",
        "content_age_days",
        "days_with_sessions",
        "days_since_last_update"
    ]
].copy()

error_details["Actual"] = wrong_predictions["Actual"]
error_details["Predicted"] = wrong_predictions["Predicted"]

display(error_details.head(3))

Incorrect predictions: 2842

Three incorrect examples:


,trend_direction,trend_pct,content_type,avg_position,content_age_days,days_with_sessions,days_since_last_update,Actual,Predicted
1,down,-57.7,keyword article,20.3,445,9,25,1,0
13,stable,10.4,keyword article,39.8,238,3,103,0,1
23,down,-100.0,keyword article,13.9,502,8,20,1,0


In [20]:
# STEP 31 — FINAL LEAKAGE AUDIT

print("Target definition:")
print("1 = trend_direction is down")
print("0 = trend_direction is not down")

print("\nForbidden columns in features:")
for col in [
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id",
    "is_declining_label"
]:
    print(f"{col}: {col in X_train.columns}")

print("\nTrain/test client overlap:",
      len(train_clients & test_clients))

Target definition:
1 = trend_direction is down
0 = trend_direction is not down

Forbidden columns in features:
trend_direction: False
trend_pct: False
content_id: False
client_id: False
is_declining_label: False

Train/test client overlap: 0


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.